In [ ]:
import * as tslab from "tslab";
import { readFileSync } from "fs";

const css = readFileSync("../style.css", "utf-8");
tslab.display.html(`<style>${css}</style>`);

# 3-Way Merge Sort: An Array-Based Implementation

The function `merge3` takes six arguments.
  - `L`      is a list,
  - `start`  is an integer such that $\texttt{start} \in \{0, \cdots, \texttt{len}(L)-1 \}$,
  - `left`   is an integer such that $\texttt{left}  \in \{0, \cdots, \texttt{len}(L)-1 \}$,
  - `right`  is an integer such that $\texttt{right} \in \{0, \cdots, \texttt{len}(L)-1 \}$,
  - `end`    is an integer such that $\texttt{end}   \in \{0, \cdots, \texttt{len}(L)-1 \}$, 
  - `A`      is a list of the same length as `L`.
  
Furthermore, the indices `start`, `left`, `right`, and `end` have to satisfy the following:
$$ 0 \leq \texttt{start} \leq \texttt{left} \leq \texttt{right} \leq \texttt{end} \leq \texttt{len}(L) $$
The function assumes that the sublists `L[start:left]`, `L[left:right]`, and `L[right:end]` are already 
sorted. The function merges these sublists so that when the call returns the sublist `L[start:end]`
is sorted.  The last argument `A` is used as auxiliary memory.

In [ ]:
function merge3(L: number[], start: number, left: number, right: number, end: number, A: number[]) {
    for (let i = start; i < end; i++) A[i] = L[i];
    let idx1 = start;
    let idx2 = left;
    let idx3 = right;
    let i = start;
    while (idx1 < left && idx2 < right && idx3 < end) {
        if (A[idx1] <= A[idx2]) {
            if (A[idx1] <= A[idx3]) {
                L[i] = A[idx1];
                idx1 += 1;
            } else {
                L[i] = A[idx3];
                idx3 += 1;
            }
        } else if (A[idx2] <= A[idx3]) {
            L[i] = A[idx2];
            idx2 += 1;
        } else {
            L[i] = A[idx3];
            idx3 += 1;
        }
        i += 1;
    }
    if (idx1 == left) {  // first list empty, merge second list and third list
        while (idx2 < right && idx3 < end) {
            if (A[idx2] <= A[idx3]) {
                L[i] = A[idx2];
                idx2 += 1;
            } else {
                L[i] = A[idx3];
                idx3 += 1;
            }
            i += 1;
        }
    } else if (idx2 == right) { // second list empty, merge first list and third list
        while (idx1 < left && idx3 < end) {
            if (A[idx1] <= A[idx3]) {
                L[i] = A[idx1];
                idx1 += 1;
            } else {
                L[i] = A[idx3];
                idx3 += 1;
            }
            i += 1;
        }
    } else if (idx3 == end) {  // third list empty, merge first list and second list
        while (idx1 < left && idx2 < right) {
            if (A[idx1] <= A[idx2]) {
                L[i] = A[idx1];
                idx1 += 1;
            } else {
                L[i] = A[idx2];
                idx2 += 1;
            }
            i += 1;
        }
    }
    if (idx1 < left) {  // second list and third list are empty
        for (let j = idx1; j < left; j++, i++) L[i] = A[j];
    }
    if (idx2 < right) { // first list and third list are empty
        for (let j = idx2; j < right; j++, i++) L[i] = A[j];
    }
    if (idx3 < end) {   // first list and second list are empty
        for (let j = idx3; j < end; j++, i++) L[i] = A[j];
    }
}

The function `mergeSort` is called with 4 arguments.
  - The first parameter `L` is the list that is to be sorted.
    However, the task of `mergeSort` is not to sort the entire list `L` but only
    the part of `L` that is given as `L[start:end]`.
  - Hence, the parameters `start` and `end` are indices specifying the 
    subarray that needs to be sorted.
  - The final parameter `A` is used as an auxiliary array.  This array is needed
    as *temporary storage* and is required to have the same size as the list `L`.

In [ ]:
function mergeSort(L: number[], start: number, end: number, A: number[]) {
    if (end - start < 2) return;
    let third = Math.floor((end - start) / 3);
    let left = start + Math.max(1, third);
    let right = start + Math.max(2, 2 * third);
    if (right > end) right = end;
    mergeSort(L, start, left, A);
    mergeSort(L, left, right, A);
    mergeSort(L, right, end, A);
    merge3(L, start, left, right, end, A);
}

The function $\texttt{sort}(L)$ sorts the list $L$ in place using *merge sort*.
It takes advantage of the fact that, in *Python*, lists are stored internally as arrays.
The function `sort` is a wrapper for the function `merge_sort`.  Its sole purpose is to allocate the auxiliary array `A`, which has the same size as the array holding `L`.

In [ ]:
function sort(L: number[]) {
    let A = L.slice();
    mergeSort(L, 0, L.length, A);
}

In [ ]:
let L = [7, 8, 11, 12, 2, 5, 3, 7, 9, 3, 2];
sort(L);
console.log(L);

## Testing

The function `counter` takes an array as input and returns a Map that keeps count of how many times each item occurs in the array.

In [ ]:
function counter<T>(arr: T[]): Map<T, number> {
  const counts = new Map<T, number>();
  for (const item of arr) {
    counts.set(item, (counts.get(item) ?? 0) + 1);
  }
  return counts;
}

We also define the helper function `compareCounter` to be able to compare the contents of two counters.

In [ ]:
function compareCounters(a: Map<number, number>, b: Map<number, number>): boolean {
  if (a.size !== b.size) return false;
  for (const [key, value] of a) {
    if (b.get(key) !== value) return false;
  }
  return true;
}

The function `isOrdered(L)` checks that the list `L` is sorted in ascending order.

In [ ]:
function isOrdered(L: number[]): void {
  for (let i = 0; i < L.length - 1; i++) {
    if (L[i] > L[i + 1]) {
      throw new Error(`${L} not ordered at ${i}`);
    }
  }
}

The function `sameElements(L, S)` returns `True`if the lists `L` and `S` contain the same elements and, furthermore, each 
element $x$ occurring in `L` occurs in `S` the same number of times it occurs in `L`.

In [ ]:
import assert from 'assert';

function sameElements(L: number[], S: number[]): void {
  assert(compareCounters(counter(L), counter(S)), "L and S do not have the same elements");
}

The function `randomIntRange(min, max)` generates a random integer in the range `[min, max)` and corresponds to Python's `range(min, max)` behavior regarding the exclusive upper bound.

In [ ]:
function randomIntRange(min: number, max: number): number {
  return Math.floor(Math.random() * (max - min)) + min;
} 

The function $\texttt{testSort}(n, k)$ generates $n$ random lists of length $k$, sorts them, and checks whether the output is sorted and contains the same elements as the input.

In [ ]:
function testSort(n: number, k: number): void {
  for (let i = 0; i < n; i++) {
    const L = Array.from({ length: k }, () => randomIntRange(0, 2 * k));
    const oldL = [...L];
    sort(L);
    isOrdered(L);
    sameElements(oldL, L);
    process.stdout.write(".");
  }
  console.log("\nAll tests successful!");
}

In [ ]:
console.time("testSort");
testSort(100, 2000);
console.timeEnd("testSort");